# V115 - Consensus override

Pre-registered plan: v114 withdrew the blanket override because top-1 flips across seeds, but rank correlation stays high, so disagreement concentrates on some users. Override only no-chat users where at least four of five specialist seeds agree on top-1, everywhere else the globalist stands. Agreement is measured out-of-fold per seed, never on test. Same slice floor and safety lock, fresh seeds. On a pass, the submission overrides exactly the consensus test users.

In [1]:
import json
import re
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold

SEEDS = [101, 202, 303, 404, 505]
N_SPLITS = 5
SLICE_FLOOR = 0.01
AGREE_MIN = 4
rng = np.random.RandomState(999)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
CHAMPION = next((p for p in [Path("/kaggle/input/intelligo-champion/submission_v42_primary.csv"),
                            Path("results/v42/submission_v42_primary.csv")] if p.exists()), None)
assert CHAMPION is not None, "full build composes onto the champion file"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v115")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve(), "| CHAMPION:", CHAMPION)

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data | CHAMPION: results/v42/submission_v42_primary.csv


## Per-seed OOF, consensus mask, gate, submission

Each seed refits the specialist per fold. Consensus counts top-1 votes. The globalist covers dissenters.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
S = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
S = np.where(np.isnan(S), np.nanmean(S, axis=0), S)
A = (S - S.mean(0)) / (S.std(0) + 1e-9)
A_tr, A_te = A[:n_train], A[n_train:]
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
is_nochat = np.array([u not in by_user for u in all_users])
nc_tr, nc_te = np.where(is_nochat[:n_train])[0], np.where(is_nochat[n_train:])[0]
print(f"no-chat train={len(nc_tr)} test={len(nc_te)}")

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

votes = np.zeros((len(nc_tr), len(MODS), len(SEEDS)))
oof_g = np.zeros((len(nc_tr), len(MODS)))
fids = np.zeros(len(nc_tr), int)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEEDS[0])
splits = list(skf.split(nc_tr, Y[nc_tr].argmax(1)))
for si, s in enumerate(SEEDS):
    for fi, (trs, vas) in enumerate(splits):
        tr_all = np.setdiff1d(np.arange(n_train), nc_tr[vas])
        tr_nc, va_nc = nc_tr[trs], nc_tr[vas]
        rb = np.random.RandomState(SEEDS[si] * 100 + fi)
        acc = np.zeros((len(va_nc), len(MODS)))
        for b in range(10):
            idx = tr_nc[rb.choice(len(tr_nc), len(tr_nc), replace=True)]
            for j in range(len(MODS)):
                t = Ridge(alpha=1.0)
                t.fit(A_tr[idx], Y[idx, j])
                acc[:, j] += t.predict(A_tr[va_nc]) / 10
        votes[vas, :, si] = acc
        for j in range(len(MODS)):
            if si == 0:
                g = Ridge(alpha=1.0)
                g.fit(A_tr[tr_all], Y[tr_all, j])
                oof_g[vas, j] = g.predict(A_tr[va_nc])
        if si == 0:
            fids[vas] = fi + 1
cons = np.zeros(len(nc_tr), int)
for i in range(len(nc_tr)):
    tops = votes[i].argmax(0)
    cons[i] = int((tops == np.bincount(tops).argmax()).sum())
print(f"consensus>=4: {(cons >= AGREE_MIN).sum()} / {len(nc_tr)}")
oof_s = votes.mean(-1)
act = oof_g.copy()
act[cons >= AGREE_MIN] = oof_s[cons >= AGREE_MIN]
sg, ss = ndcg5(Y[nc_tr], oof_g), ndcg5(Y[nc_tr], act)
d = ss - sg
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(5000)], [0.025, 0.975])
wf = min((np.mean(ss[fids == k]) - np.mean(sg[fids == k])) for k in range(1, N_SPLITS + 1))
print(f"globalist={sg.mean():.5f} consensus={ss.mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
ok = (lo > SLICE_FLOOR) and (wf >= 0)
print("ADOPT consensus override" if ok else "KEEP globalist; consensus does not stabilize the gain")

ch = pd.read_csv(CHAMPION)
assert ch["user_id"].tolist() == test_ids
C = ch[MODS].to_numpy(float)
report = {"adopt": bool(ok)}
final = C.copy()
if ok:
    P = np.zeros((len(nc_te), len(MODS)))
    for j in range(len(MODS)):
        s = Ridge(alpha=1.0)
        s.fit(A_tr[nc_tr], Y[nc_tr, j])
        P[:, j] = s.predict(A_te[nc_te])
    T = np.zeros((len(nc_te), len(SEEDS)), int)
    for si, s in enumerate(SEEDS):
        r = np.random.RandomState(s)
        acc = np.zeros_like(P)
        for b in range(10):
            idx = nc_tr[r.choice(len(nc_tr), len(nc_tr), replace=True)]
            for j in range(len(MODS)):
                m = Ridge(alpha=1.0)
                m.fit(A_tr[idx], Y[idx, j])
                acc[:, j] += m.predict(A_te[nc_te]) / 10
        T[:, si] = acc.argmax(1)
    agree = (T == np.apply_along_axis(lambda r: np.bincount(r).argmax(), 1, T)[:, None]).sum(1)
    keep = nc_te[agree >= AGREE_MIN]
    S = C.copy()
    S[keep] = P[np.isin(nc_te, keep)]
    changed = set(np.where((S != C).any(1))[0].tolist())
    assert changed <= set(keep.tolist()), "geometry breach"
    report.update({"changed": len(changed), "top1_changes": int((C.argmax(1) != S.argmax(1)).sum())})
    print(f"geometry: changed={len(changed)} top1-changes={report['top1_changes']}")
    final = S
else:
    print("FAIL-CLOSE: champion untouched")
sub = pd.DataFrame(final, columns=MODS)
sub.insert(0, "user_id", test_ids)
sub.to_csv(OUT / "submission_v115_primary.csv", index=False)
report["sha"] = sha256(pd.util.hash_pandas_object(sub).values.tobytes()).hexdigest()[:16]
report["champion_sha"] = sha256(pd.util.hash_pandas_object(ch).values.tobytes()).hexdigest()[:16]
Path(OUT / "v115_gate_report.json").write_text(json.dumps(report, indent=1))
print("wrote", OUT / "submission_v115_primary.csv", report)

no-chat train=117 test=33


/home/athilla/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


consensus>=4: 94 / 117
globalist=0.52586 consensus=0.58127 delta=+0.05542 CI95=[+0.02090,+0.08935] worst-fold=+0.01447
ADOPT consensus override


geometry: changed=23 top1-changes=7
wrote results/v115/submission_v115_primary.csv {'adopt': True, 'changed': 23, 'top1_changes': 7, 'sha': '8e4198e69513873a', 'champion_sha': '7bc841d13df9f947'}
